In [2]:
#!/usr/bin/env python3
"""
================================================================================
DIAGNOSI OUTLIER -- confronto dettagliato am/dc/n3DP su Z_true vs Z_gnn
================================================================================
Per le istanze con gap_ML > 10%, ricalcola sia la soluzione esatta (CG+MILP)
sia la pipeline GNN, estraendo il dettaglio delle decisioni (non solo il
costo) per capire ESATTAMENTE dove divergono: quali SKU sono am nell'una e
cm nell'altra, e se n3DP differisce (verifica l'ipotesi del meccanismo di
co-ammortamento a soglia).
================================================================================
"""
import json, glob, os, time
import numpy as np
import torch
import gurobipy as gp
from gurobipy import GRB

# Importa tutte le funzioni gia' scritte in optimality_gap_v2.py
import sys
sys.path.insert(0, os.getcwd())
from optimality_gap_v2 import (
    inst_from_json, column_generation, build_milp, get_time_limit,
    repair_predictions, load_instance_graph, HeteroGNN,
    HIDDEN_DIM, NUM_LAYERS, DROPOUT, THREADS_PER_SOLVE, MIP_GAP_TARGET,
    DATASET_DIR, MODEL_PATH
)

OUTLIER_IDS = [160, 178, 42, 395, 99, 485, 265]  # <-- adatta se ne vuoi altri


def solve_free_detailed(inst, warm_start, n_threads, time_limit, mip_gap):
    """Come solve_free originale, ma espone am, dc, n3DP completi."""
    mdl, v, _ = build_milp(inst, n_threads, time_limit, mip_gap)
    ws = warm_start
    nI, nD, nC = inst['nI'], inst['nD'], inst['nC']
    for i in range(nI):
        v['am'][i].Start = float(ws['am'][i]); v['cm'][i].Start = float(1-ws['am'][i])
    for d in range(nD):
        for i in range(nI):
            v['dc'][d,i].Start = float(ws['dc'][d,i])
            v['v'][i,d].Start = float(ws['am'][i] and ws['dc'][d,i])
            v['w'][i,d].Start = float((not ws['am'][i]) and ws['dc'][d,i])
            for c in range(nC): v['y'][d,i,c].Start = float(ws['y'][d,i,c]>0.5)
        v['n3DP'][d].Start = float(ws['n3dp'][d])
    mdl.update(); mdl.optimize()
    if mdl.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL] or mdl.SolCount == 0:
        return None
    return {
        'obj': mdl.ObjVal, 'gap': mdl.MIPGap*100,
        'am': np.array([v['am'][i].X>0.5 for i in range(nI)]),
        'dc': np.array([[v['dc'][d,i].X>0.5 for i in range(nI)] for d in range(nD)]),
        'n3DP': np.array([v['n3DP'][d].X for d in range(nD)]),
    }


def solve_fixed_detailed(inst, am_pred, dc_pred, y_pred, n_threads, time_limit=600):
    """Come solve_fixed originale, ma espone anche n3DP risolto."""
    mdl, v, _ = build_milp(inst, n_threads, time_limit, mip_gap=0.001)
    nI, nD, nC = inst['nI'], inst['nD'], inst['nC']
    for i in range(nI):
        v['am'][i].lb = v['am'][i].ub = float(am_pred[i])
        v['cm'][i].lb = v['cm'][i].ub = float(1 - am_pred[i])
    for d in range(nD):
        for i in range(nI):
            v['dc'][d,i].lb = v['dc'][d,i].ub = float(dc_pred[d,i])
            v['v'][i,d].lb = v['v'][i,d].ub = float(am_pred[i] and dc_pred[d,i])
            v['w'][i,d].lb = v['w'][i,d].ub = float((1-am_pred[i]) and dc_pred[d,i])
            for c in range(nC):
                v['y'][d,i,c].lb = v['y'][d,i,c].ub = float(y_pred[d,i,c])
    mdl.update(); mdl.optimize()
    if mdl.Status not in [GRB.OPTIMAL,GRB.TIME_LIMIT,GRB.SUBOPTIMAL] or mdl.SolCount==0:
        return None
    return {'obj': mdl.ObjVal, 'gap': mdl.MIPGap*100,
            'n3DP': np.array([v['n3DP'][d].X for d in range(nD)])}


if __name__ == '__main__':
    files = sorted(glob.glob(os.path.join(DATASET_DIR, "instance_*.json")))
    id_to_file = {}
    for fp in files:
        with open(fp) as fh:
            d = json.load(fh)
        # Forza la conversione a intero per evitare conflitti stringa/numero
        id_to_file[int(d['instance_id'])] = fp

    ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
    model = HeteroGNN(HIDDEN_DIM, NUM_LAYERS, DROPOUT)
    model.load_state_dict(ckpt['model_state_dict']); model.eval()

    for iid in OUTLIER_IDS:
        if iid not in id_to_file:
            print(f"#{iid}: file non trovato, skip"); continue
        with open(id_to_file[iid]) as fh:
            d = json.load(fh)
        nI, nD, nC = d['nI'], d['nD'], d['nC']
        inst = inst_from_json(d)
        print(f"\n{'='*70}\n#{iid} (nI={nI}, nD={nD}, nC={nC})\n{'='*70}")

        t0 = time.time()
        ws = column_generation(inst, n_threads=THREADS_PER_SOLVE)
        res_true = solve_free_detailed(inst, ws, n_threads=THREADS_PER_SOLVE,
                                        time_limit=get_time_limit(nI), mip_gap=MIP_GAP_TARGET)
        print(f"  Z_true risolto in {time.time()-t0:.1f}s")
        if res_true is None:
            print("  Z_true INFEASIBLE, skip"); continue

        graph = load_instance_graph(d)
        with torch.no_grad():
            x_dict = model(graph)
            am_prob = torch.sigmoid(model.predict_am(x_dict)).numpy()
            dc_prob = torch.sigmoid(model.predict_dc(x_dict, nI, nD)).numpy()
            y_logits = model.predict_y(x_dict, graph, nI, nD, nC).numpy()
        am_pred, dc_pred, y_pred = repair_predictions(am_prob, dc_prob, y_logits, nI, nD, nC)
        res_gnn = solve_fixed_detailed(inst, am_pred, dc_pred, y_pred, n_threads=THREADS_PER_SOLVE)
        if res_gnn is None:
            print("  Z_gnn INFEASIBLE, skip"); continue

        gap_pct = 100*(res_gnn['obj']-res_true['obj'])/res_true['obj']
        am_true, am_gnn = res_true['am'].astype(int), am_pred.astype(int)
        diff_sku = np.where(am_true != am_gnn)[0]

        print(f"  Z_true={res_true['obj']:,.0f}  Z_gnn={res_gnn['obj']:,.0f}  gap={gap_pct:+.2f}%")
        print(f"  n3DP vero: {res_true['n3DP'].round(2).tolist()}   n3DP GNN: {res_gnn['n3DP'].round(2).tolist()}")
        print(f"  n_AM vero: {am_true.sum()}/{nI}   n_AM GNN: {am_gnn.sum()}/{nI}")
        if len(diff_sku) == 0:
            print("  am identici su tutti gli SKU -- la differenza NON e' nella scelta tecnologica "
                  "(probabile causa: dc/y diversi, o n3DP diverso a parita' di am)")
        else:
            print(f"  SKU con am DIVERSO ({len(diff_sku)}): {diff_sku.tolist()}")
            for i in diff_sku:
                ratio_i = d['ratio'][i]
                print(f"    SKU {i}: ratio={ratio_i:.3f}  vero={'AM' if am_true[i] else 'CM'}  "
                      f"GNN={'AM' if am_gnn[i] else 'CM'}")
        n3dp_diff = not np.allclose(res_true['n3DP'], res_gnn['n3DP'], atol=0.01)
        print(f"  n3DP DIVERSO tra vero e GNN: {n3dp_diff}")

#160: file non trovato, skip
#178: file non trovato, skip
#42: file non trovato, skip
#395: file non trovato, skip
#99: file non trovato, skip
#485: file non trovato, skip
#265: file non trovato, skip


In [ ]:
#!/usr/bin/env python3
"""
================================================================================
DIAGNOSI OUTLIER -- confronto dettagliato am/dc/n3DP su Z_true vs Z_gnn
================================================================================
Per le istanze con gap_ML > 10%, ricalcola sia la soluzione esatta (CG+MILP)
sia la pipeline GNN, estraendo il dettaglio delle decisioni (non solo il
costo) per capire ESATTAMENTE dove divergono: quali SKU sono am nell'una e
cm nell'altra, e se n3DP differisce (verifica l'ipotesi del meccanismo di
co-ammortamento a soglia).
================================================================================
"""
import json, glob, os, time
import numpy as np
import torch
import gurobipy as gp
from gurobipy import GRB

# Importa tutte le funzioni gia' scritte in optimality_gap_v2.py
import sys
sys.path.insert(0, os.getcwd())
from optimality_gap_v2 import (
    inst_from_json, column_generation, build_milp, get_time_limit,
    repair_predictions, load_instance_graph, HeteroGNN,
    HIDDEN_DIM, NUM_LAYERS, DROPOUT, THREADS_PER_SOLVE, MIP_GAP_TARGET,
)


DATASET_DIR = r"C:\Users\stefa\OneDrive\Desktop\MAGISTRALE\TESI\TEST CODICE\GNN dataset\gnn_dataset_v2_TEST"
MODEL_PATH  = r"C:\Users\stefa\OneDrive\Desktop\MAGISTRALE\TESI\TEST CODICE\GNN dataset\gnn_model.pt"  # verifica il nome esatto del file

OUTLIER_IDS = [160, 178, 42, 395, 99, 485, 265]  # <-- adatta se ne vuoi altri


def solve_free_detailed(inst, warm_start, n_threads, time_limit, mip_gap):
    """Come solve_free originale, ma espone am, dc, n3DP completi."""
    mdl, v, _ = build_milp(inst, n_threads, time_limit, mip_gap)
    ws = warm_start
    nI, nD, nC = inst['nI'], inst['nD'], inst['nC']
    for i in range(nI):
        v['am'][i].Start = float(ws['am'][i]); v['cm'][i].Start = float(1-ws['am'][i])
    for d in range(nD):
        for i in range(nI):
            v['dc'][d,i].Start = float(ws['dc'][d,i])
            v['v'][i,d].Start = float(ws['am'][i] and ws['dc'][d,i])
            v['w'][i,d].Start = float((not ws['am'][i]) and ws['dc'][d,i])
            for c in range(nC): v['y'][d,i,c].Start = float(ws['y'][d,i,c]>0.5)
        v['n3DP'][d].Start = float(ws['n3dp'][d])
    mdl.update(); mdl.optimize()
    if mdl.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL] or mdl.SolCount == 0:
        return None
    return {
        'obj': mdl.ObjVal, 'gap': mdl.MIPGap*100,
        'am': np.array([v['am'][i].X>0.5 for i in range(nI)]),
        'dc': np.array([[v['dc'][d,i].X>0.5 for i in range(nI)] for d in range(nD)]),
        'n3DP': np.array([v['n3DP'][d].X for d in range(nD)]),
    }


def solve_fixed_detailed(inst, am_pred, dc_pred, y_pred, n_threads, time_limit=600):
    """Come solve_fixed originale, ma espone anche n3DP risolto."""
    mdl, v, _ = build_milp(inst, n_threads, time_limit, mip_gap=0.001)
    nI, nD, nC = inst['nI'], inst['nD'], inst['nC']
    for i in range(nI):
        v['am'][i].lb = v['am'][i].ub = float(am_pred[i])
        v['cm'][i].lb = v['cm'][i].ub = float(1 - am_pred[i])
    for d in range(nD):
        for i in range(nI):
            v['dc'][d,i].lb = v['dc'][d,i].ub = float(dc_pred[d,i])
            v['v'][i,d].lb = v['v'][i,d].ub = float(am_pred[i] and dc_pred[d,i])
            v['w'][i,d].lb = v['w'][i,d].ub = float((1-am_pred[i]) and dc_pred[d,i])
            for c in range(nC):
                v['y'][d,i,c].lb = v['y'][d,i,c].ub = float(y_pred[d,i,c])
    mdl.update(); mdl.optimize()
    if mdl.Status not in [GRB.OPTIMAL,GRB.TIME_LIMIT,GRB.SUBOPTIMAL] or mdl.SolCount==0:
        return None
    return {'obj': mdl.ObjVal, 'gap': mdl.MIPGap*100,
            'n3DP': np.array([v['n3DP'][d].X for d in range(nD)])}


if __name__ == '__main__':
    files = sorted(glob.glob(os.path.join(DATASET_DIR, "instance_*.json")))
    id_to_file = {}
    for fp in files:
        with open(fp) as fh:
            d = json.load(fh)
        # Forza la conversione a intero per evitare conflitti stringa/numero
        id_to_file[int(d['instance_id'])] = fp

    print(f"File trovati in DATASET_DIR: {len(files)}")
    print(f"Istanze indicizzate: {len(id_to_file)}")
    if len(id_to_file) == 0:
        print(f"ATTENZIONE: nessun file trovato in {DATASET_DIR} -- verifica il percorso!")
    else:
        print(f"Esempio ID disponibili: {sorted(id_to_file.keys())[:10]}")

    ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
    model = HeteroGNN(HIDDEN_DIM, NUM_LAYERS, DROPOUT)
    model.load_state_dict(ckpt['model_state_dict']); model.eval()

    for iid in OUTLIER_IDS:
        if iid not in id_to_file:
            print(f"#{iid}: file non trovato, skip"); continue
        with open(id_to_file[iid]) as fh:
            d = json.load(fh)
        nI, nD, nC = d['nI'], d['nD'], d['nC']
        inst = inst_from_json(d)
        print(f"\n{'='*70}\n#{iid} (nI={nI}, nD={nD}, nC={nC})\n{'='*70}")

        t0 = time.time()
        ws = column_generation(inst, n_threads=THREADS_PER_SOLVE)
        res_true = solve_free_detailed(inst, ws, n_threads=THREADS_PER_SOLVE,
                                        time_limit=get_time_limit(nI), mip_gap=MIP_GAP_TARGET)
        print(f"  Z_true risolto in {time.time()-t0:.1f}s")
        if res_true is None:
            print("  Z_true INFEASIBLE, skip"); continue

        graph = load_instance_graph(d)
        with torch.no_grad():
            x_dict = model(graph)
            am_prob = torch.sigmoid(model.predict_am(x_dict)).numpy()
            dc_prob = torch.sigmoid(model.predict_dc(x_dict, nI, nD)).numpy()
            y_logits = model.predict_y(x_dict, graph, nI, nD, nC).numpy()
        am_pred, dc_pred, y_pred = repair_predictions(am_prob, dc_prob, y_logits, nI, nD, nC)
        res_gnn = solve_fixed_detailed(inst, am_pred, dc_pred, y_pred, n_threads=THREADS_PER_SOLVE)
        if res_gnn is None:
            print("  Z_gnn INFEASIBLE, skip"); continue

        gap_pct = 100*(res_gnn['obj']-res_true['obj'])/res_true['obj']
        am_true, am_gnn = res_true['am'].astype(int), am_pred.astype(int)
        diff_sku = np.where(am_true != am_gnn)[0]

        print(f"  Z_true={res_true['obj']:,.0f}  Z_gnn={res_gnn['obj']:,.0f}  gap={gap_pct:+.2f}%")
        print(f"  n3DP vero: {res_true['n3DP'].round(2).tolist()}   n3DP GNN: {res_gnn['n3DP'].round(2).tolist()}")
        print(f"  n_AM vero: {am_true.sum()}/{nI}   n_AM GNN: {am_gnn.sum()}/{nI}")
        if len(diff_sku) == 0:
            print("  am identici su tutti gli SKU -- la differenza NON e' nella scelta tecnologica "
                  "(probabile causa: dc/y diversi, o n3DP diverso a parita' di am)")
        else:
            print(f"  SKU con am DIVERSO ({len(diff_sku)}): {diff_sku.tolist()}")
            for i in diff_sku:
                ratio_i = d['ratio'][i]
                print(f"    SKU {i}: ratio={ratio_i:.3f}  vero={'AM' if am_true[i] else 'CM'}  "
                      f"GNN={'AM' if am_gnn[i] else 'CM'}")
        n3dp_diff = not np.allclose(res_true['n3DP'], res_gnn['n3DP'], atol=0.01)
        print(f"  n3DP DIVERSO tra vero e GNN: {n3dp_diff}")

File trovati in DATASET_DIR: 608
Istanze indicizzate: 608
Esempio ID disponibili: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

#160 (nI=8, nD=3, nC=5)
Set parameter Username
Set parameter LicenseID to value 2800035
Academic license - for non-commercial use only - expires 2027-03-30
  Z_true risolto in 619.4s
  Z_true=124,572  Z_gnn=176,757  gap=+41.89%
  n3DP vero: [-0.0, -0.0, -0.0]   n3DP GNN: [1.0, -0.0, 1.0]
  n_AM vero: 0/8   n_AM GNN: 2/8
  SKU con am DIVERSO (2): [2, 4]
    SKU 2: ratio=0.976  vero=CM  GNN=AM
    SKU 4: ratio=1.157  vero=CM  GNN=AM
  n3DP DIVERSO tra vero e GNN: True

#178 (nI=10, nD=3, nC=5)
  Z_true risolto in 47.4s
  Z_true=107,906  Z_gnn=137,867  gap=+27.77%
  n3DP vero: [0.0, -0.0, 1.0]   n3DP GNN: [1.0, -0.0, 1.0]
  n_AM vero: 3/10   n_AM GNN: 3/10
  am identici su tutti gli SKU -- la differenza NON e' nella scelta tecnologica (probabile causa: dc/y diversi, o n3DP diverso a parita' di am)
  n3DP DIVERSO tra vero e GNN: True

#42 (nI=6, nD=2, nC=4)
  Z_true risolto in 